In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/hourly_cleaned.csv")

df["interval_start"] = pd.to_datetime(
    df["interval_start"],
    errors="coerce"
)

df = df.sort_values(
    ["equipment_ID", "interval_start"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)

Dataset shape: (23376, 164)


In [25]:
required_columns = [
    "equipment_ID",
    "interval_start",
    "%production",
    "%downtime",
    "%idle",
    "%performance_loss"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

All required columns are available.


In [26]:
production_rolling_6h = (
    df.groupby("equipment_ID")["%production"]
      .transform(
          lambda x: x.rolling(
              window=6,
              min_periods=1
          ).mean()
      )
)

downtime_rolling_6h = (
    df.groupby("equipment_ID")["%downtime"]
      .transform(
          lambda x: x.rolling(
              window=6,
              min_periods=1
          ).mean()
      )
)

In [27]:
production_change = (
    df.groupby("equipment_ID")["%production"]
      .diff()
)

downtime_change = (
    df.groupby("equipment_ID")["%downtime"]
      .diff()
)

performance_change = (
    df.groupby("equipment_ID")["%performance_loss"]
      .diff()
)

In [28]:
health_score = (
    100
    - df["%downtime"].fillna(0)
    - df["%idle"].fillna(0)
    - df["%performance_loss"].fillna(0)
)

health_score = health_score.clip(0, 100)

In [29]:
new_features = pd.DataFrame({
    "production_rolling_6h": production_rolling_6h,
    "downtime_rolling_6h": downtime_rolling_6h,
    "production_change": production_change,
    "downtime_change": downtime_change,
    "performance_change": performance_change,
    "health_score": health_score
})

df = pd.concat(
    [df, new_features],
    axis=1
)

df = df.copy()

print("Feature engineering completed.")
print("New dataset shape:", df.shape)

Feature engineering completed.
New dataset shape: (23376, 170)


In [30]:
feature_columns = [
    "production_rolling_6h",
    "downtime_rolling_6h",
    "production_change",
    "downtime_change",
    "performance_change",
    "health_score"
]

df[feature_columns].head(10)

,production_rolling_6h,downtime_rolling_6h,production_change,downtime_change,performance_change,health_score
0,0.861729,0.052261,NaN,NaN,NaN,99.906124
1,0.866313,0.084982,0.009168,0.065443,-0.008692,99.870897
2,0.903703,0.063827,0.107586,-0.096186,-0.011399,99.978483
3,0.795544,0.164966,-0.507417,0.446867,0.000000,99.531616
4,0.833840,0.133529,0.515958,-0.460606,0.005198,99.987024
5,0.811000,0.161807,-0.290223,0.295420,-0.005198,99.696801
6,0.729474,0.257668,-0.324228,0.324228,0.000000,99.372573
7,0.702908,0.274859,0.338927,-0.406582,0.002757,99.776398
8,0.705173,0.272266,0.280573,-0.214883,-0.000791,99.992072
9,0.773965,0.200549,-0.108254,0.032121,-0.001966,99.961917


In [31]:
change_columns = [
    "production_change",
    "downtime_change",
    "performance_change"
]

df[change_columns] = df[change_columns].fillna(0)

In [32]:
print("Missing values in engineered features:")
print(df[feature_columns].isnull().sum())

Missing values in engineered features:
production_rolling_6h    0
downtime_rolling_6h      0
production_change        0
downtime_change          0
performance_change       0
health_score             0
dtype: int64


In [33]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/features.csv",
    index=False
)

print("features.csv saved successfully!")
print("Final shape:", df.shape)

features.csv saved successfully!
Final shape: (23376, 170)


In [1]:
import pandas as pd

features_df = pd.read_csv(
    "../Data/processed/features.csv"
)

print(features_df.shape)

(23376, 170)


In [2]:
print(features_df["equipment_ID"].unique())

<ArrowStringArray>
['s_1', 's_2', 's_3', 's_4', 's_5']
Length: 5, dtype: str


In [3]:
print(
    features_df["equipment_ID"]
    .astype(str)
    .map(repr)
    .unique()
)

<ArrowStringArray>
[''s_1'', ''s_2'', ''s_3'', ''s_4'', ''s_5'']
Length: 5, dtype: str


In [4]:
print(
    features_df[
        features_df["equipment_ID"]
        .astype(str)
        .str.strip()
        == "A_001"
    ].shape
)

(0, 170)
